In [1]:
import codecs
import numpy as np
import gensim
import textwrap
import torch

In [2]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("all cool")
else:
    print("плак плак")

all cool


In [3]:
torch.cuda.empty_cache()

In [4]:
with open(
    "/home/julia/Desktop/code/PAC/RNN/измененные андроиды.txt",
    "r",
    encoding="utf-8",
) as f:
    docs = f.readlines()

max_sentence_len = 12

sentences = [sent for doc in docs for sent in doc.split(".")]
sentences = [[word for word in sent.lower().split()[:max_sentence_len]] for sent in sentences]
sentences = [sent for sent in sentences if len(sent) > 1]
print(len(sentences), "предложений")

# Обучение модели
word_model = gensim.models.Word2Vec(
    sentences, vector_size=100, min_count=1, window=5, epochs=100
)

pretrained_weights = word_model.wv.vectors
vocab_size, embedding_size = pretrained_weights.shape
print(vocab_size, embedding_size)

print("Похожие слова:")
for word in ["андроид", "рик", "декард", "тест", "сова", "паук"]:
    most_similar = ", ".join(
        "%s (%.2f)" % (similar, dist)
        for similar, dist in word_model.wv.most_similar(word)[:8]
    )
    print("  %s -> %s" % (word, most_similar))

6104 предложений
14128 100
Похожие слова:
  андроид -> прилетевший (0.57), скрывается (0.56), точностью (0.54), вызывает (0.54), сбежавший (0.53), признайтесь (0.52), жаждущий (0.52), вытащили (0.52)
  рик -> он (0.56), взаимно (0.47), увидимся (0.46), норм (0.43), натянуто (0.42), мышь (0.42), угрюмо (0.42), существу (0.42)
  декард -> спецэффектами (0.72), уэйд (0.72), голливуде (0.69), корпорацию (0.69), незавидное (0.68), снимки (0.64), мерривелл (0.63), поздравления (0.61)
  тест -> войткампфа (0.75), эмпатию (0.70), фиксируют (0.70), шкала (0.70), провести (0.68), проводят (0.68), профильный (0.66), бонелли (0.66)
  сова -> корпорация (0.64), официально (0.64), элдон (0.64), дремавшую (0.60), свирепо (0.59), поддельная (0.59), помни (0.59), пятно (0.57)
  паук -> ползал (0.63), живой (0.62), ненужные (0.62), последним (0.55), калека (0.54), ярком (0.53), покачивался (0.52), захлестнуло (0.52)


и снова я задалась вопросом что здесь х, у для модели
<br>
ответ китайского разума
<br>
Вариант "Векторизованный" (чаще используется для обучения):<br>
Мы берем всю последовательность кроме последнего слова как вход, и всю последовательность без первого слова как цель.<br>

    X = [["рик"], ["декард"], ["убил"]] (последовательность длины 3)<br>
    Y = ["декард", "убил", "андроидов"] (последовательность длины 3)<br>

В тензорном виде это выглядит так:<br>

    X: матрица, где строки — это векторы слов "рик", "декард", "убил".<br>
    Y: вектор индексов слов ["декард", "убил", "андроидов"]<br>

In [5]:
# Для каждого предложения создаём несколько пар (X, Y)
# Пример: ["рик", "декард", "убил"] ->
#   X=[рик], Y=декард
#   X=[рик, декард], Y=убил

In [6]:
word2idx = {word: idx for idx, word in enumerate(word_model.wv.index_to_key)}
idx2word = {idx: word for word, idx in word2idx.items()}

In [7]:
vocab_size = len(word2idx)  
print(f"Размер словаря: {vocab_size}")

Размер словаря: 14128


In [8]:
X = []
Y = []

for sent in sentences:
    if (len(sent) <2):
        continue
    for i in range(1, len(sent)):
        # X - последовательность векторов слов до текущего
        x_vectors = [word_model.wv[sent[j]] for j in range(i)]
        # Y - индекс следующего слова
        y_idx = word2idx[sent[i]]
        if len(x_vectors) < max_sentence_len:
            x_vectors = x_vectors + [np.zeros(embedding_size)] * (max_sentence_len - len(x_vectors))

        X.append(x_vectors)
        Y.append(y_idx)

X = (torch.FloatTensor(np.array(X))).to(device)
Y = (torch.LongTensor(Y)).to(device)

In [9]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X,Y)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)


In [10]:
import torch.nn as nn
from torch import optim

class ElSheepModel(nn.Module):
    def __init__(self, input_size=100, hidden_size=300, num_layers=2):
        super().__init__()
        # input_size должен совпадать с размером алфавита
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        # Выходной слой должен выдавать размер алфавита
        self.dense = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        # LSTM возвращает (output_for_all_steps, (hidden_state, cell_state))
        # Мы берем только output, hidden нам не нужен явно
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        # Применяем линейный слой к каждому шагу последовательности
        out = self.dense(out)
        return out

model = ElSheepModel().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=10
)


epochs = 100

best_loss = 100000
patience_count = 0

for epoch in range(epochs):
    epoch_loss = 0
    num_batches = 0

    for batch_X, batch_Y in dataloader:
        optimizer.zero_grad()

        output = model(batch_X)

        loss = criterion(output, batch_Y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    avg_loss = epoch_loss / num_batches
    scheduler.step(avg_loss)
    if avg_loss <= best_loss:
        best_loss = avg_loss
        patience_count = 0
    else:
        if patience_count < 5:
            patience_count += 1
        else:
            print("early stop")
            print(f"Эпоха {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
            break
    print(f"Эпоха {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

Эпоха 1/100, Loss: 8.0806
Эпоха 2/100, Loss: 7.7391
Эпоха 3/100, Loss: 7.6111
Эпоха 4/100, Loss: 7.3917
Эпоха 5/100, Loss: 7.1041
Эпоха 6/100, Loss: 6.7429
Эпоха 7/100, Loss: 6.2864
Эпоха 8/100, Loss: 5.7935
Эпоха 9/100, Loss: 5.2579
Эпоха 10/100, Loss: 4.7067
Эпоха 11/100, Loss: 4.1566
Эпоха 12/100, Loss: 3.6325
Эпоха 13/100, Loss: 3.1672
Эпоха 14/100, Loss: 2.7598
Эпоха 15/100, Loss: 2.4042
Эпоха 16/100, Loss: 2.0936
Эпоха 17/100, Loss: 1.8295
Эпоха 18/100, Loss: 1.5967
Эпоха 19/100, Loss: 1.3974
Эпоха 20/100, Loss: 1.2165
Эпоха 21/100, Loss: 1.0578
Эпоха 22/100, Loss: 0.9301
Эпоха 23/100, Loss: 0.8194
Эпоха 24/100, Loss: 0.7323
Эпоха 25/100, Loss: 0.6582
Эпоха 26/100, Loss: 0.5992
Эпоха 27/100, Loss: 0.5534
Эпоха 28/100, Loss: 0.5159
Эпоха 29/100, Loss: 0.4880
Эпоха 30/100, Loss: 0.4642
Эпоха 31/100, Loss: 0.4449
Эпоха 32/100, Loss: 0.4286
Эпоха 33/100, Loss: 0.4144
Эпоха 34/100, Loss: 0.4127
Эпоха 35/100, Loss: 0.3972
Эпоха 36/100, Loss: 0.3855
Эпоха 37/100, Loss: 0.3754
Эпоха 38/1

In [11]:
def generate_text(
    prompt_words,
    model,
    word_model,
    word2idx,
    idx2word,
    max_len=12,
    num_words=10,
    temperature=0.8,
):
    model.eval() 

    current_words = prompt_words.copy()
    generated = []

    with torch.no_grad():
        for _ in range(num_words):
            # === 1. Подготовка входа ===
            # Берем последние max_len-1 слов (потому что одно место оставим для нового)
            context = (
                current_words[-(max_len - 1) :]
                if len(current_words) >= max_len
                else current_words
            )

            # Конвертируем слова в векторы Word2Vec
            x_vectors = [
                word_model.wv[word] for word in context if word in word_model.wv
            ]

            # Если слов нет — используем нулевые векторы
            if len(x_vectors) == 0:
                x_vectors = [np.zeros(embedding_size)]

            # Добавляем padding до max_len
            if len(x_vectors) < max_len:
                padding = [np.zeros(embedding_size)] * (max_len - len(x_vectors))
                x_vectors = padding + x_vectors  # Padding в начало

            # Конвертируем в тензор
            x_tensor = torch.FloatTensor(np.array([x_vectors])).to(
                device
            )  # (1, max_len, 100)

            # === 2. Предсказание ===
            output = model(x_tensor)  # (1, vocab_size)
            probs = torch.softmax(output / temperature, dim=-1)[0]  # Вероятности

            # === 3. Выбор следующего слова ===
            # Вариант 1: брать наиболее вероятное (детерминировано)
            # next_idx = torch.argmax(probs).item()

            # Вариант 2: сэмплировать из распределения (более интересно)
            next_idx = torch.multinomial(probs, 1).item()

            next_word = idx2word.get(next_idx, "<UNK>")

            # === 4. Добавляем слово ===
            generated.append(next_word)
            current_words.append(next_word)

    model.train()  # Возвращаем режим обучения
    return " ".join(prompt_words + generated)

In [14]:
# === Тестируем генерацию ===
print("\n=== Генерация текста ===")

# Промпты для теста
prompts = [
    ["рик"],
    ["андроид"],
    ["тест"],
]

for prompt in prompts:
    generated = generate_text(
        prompt_words=prompt,
        model=model,
        word_model=word_model,
        word2idx=word2idx,
        idx2word=idx2word,
        max_len=max_sentence_len,
        num_words=8,
        temperature=0.9,
    )
    print(f"\nПромпт: {' '.join(prompt)}")
    print(f"Результат: {generated}")


=== Генерация текста ===

Промпт: рик
Результат: рик ее к эмпатоскопом она аэратор большого печальных огромным

Промпт: андроид
Результат: андроид и кивнул на сверчка него что на холла

Промпт: тест
Результат: тест вам и и сконцентрировавшись ее бы чертовски подозрения
